<a href="https://colab.research.google.com/github/Annpeng1005/MLH-Crypto-Tracker/blob/main/Sprint_04_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 4: RAG Pipeline (Retrieval-Augmented Generation)

Welcome to the final boss of your MLH project!

Currently, our AI is "blind." It knows the price dropped, but it doesn't know *why*. Today, we are going to give it eyes by fetching live news articles and feeding them directly into our prompt.

This technique is called **RAG**, and it is how modern PMs build intelligent features that don't hallucinate.

---

## The Problem (The "Why")
When a coin crashes, we need to automatically read the news to find out why. We will use an API from CryptoCompare that returns news articles based on a coin's symbol.

Because cloud servers share IP addresses, CryptoCompare will block anonymous requests from Colab. You need an API key to prove you are an authenticated developer.

---

## Your Acceptance Criteria

1. **Get the Key:** Go to [CryptoCompare](https://www.cryptocompare.com/cryptopian/api-keys), create a free account, and get an API key. Save it in your Colab Secrets tab exactly as `CRYPTO_API_KEY`.
2. **Defensive Setup:** Write an `if not crypto_key:` statement right after you load it. If the key is empty, use `raise ValueError("Key is missing!")` to crash the script safely rather than pinging the API with a blank key.
3. **Construct the Header:** In production environments, it is a security risk to put API keys directly in the URL. Instead, we use HTTP Headers.
   * Create a dictionary: `headers = {"Authorization": f"Apikey {crypto_key}"}` *(Note the capital 'A' in Authorization!)*
4. **Trigger the News:** Inside your existing `if daily_change <= -5.0:` block, make your request using a 2-second pause (`time.sleep(2)`) to avoid rate limits.
   * *URL format:* `f"https://min-api.cryptocompare.com/data/v2/news/?lang=EN&categories={coin_symbol}"`
   * *The Call:* `requests.get(news_url, headers=headers).json()`
5. **Upgrade the Prompt:** Rewrite your Gemini prompt. Pass in the coin name, the price drop, AND the 3 headlines. Ask Gemini to write a 2-sentence Slack alert explaining the drop using the provided news context.
6. **Export:** Append this new alert to your `market_alerts.txt` file.

---

## Final Version Control
Once your AI is accurately summarizing the news, commit and push your final notebook to your GitHub repository. You now have a complete, AI-powered data engineering portfolio piece ready for your MLH application!

In [1]:
# Install your AI library here (e.g., !pip install openai)

# Build your final RAG pipeline below!
!pip install -U google-genai

import requests
import time

from google import genai






In [2]:
from google.colab import userdata
Gemini_api_key = userdata.get('mohammad_tier_1_gemini_api_key')
Crypto_api_key = userdata.get('crypto_api_key')

#not sure about this part - from chat
headers = {"Authorization": f"Apikey {Crypto_api_key}"}






In [3]:
from google import genai

# Initialize the client
# Replace 'YOUR_API_KEY' with your actual key or use environment variables
client = genai.Client(api_key= Gemini_api_key)

# Generate content
response = client.models.generate_content(
    model="gemini-2.5-flash", # Use the latest available model
    contents="Explain quantum computing in one sentence."
)
print(response.text)
  # for model in client.models.list():
        # print(model.name)

Quantum computing leverages the principles of quantum mechanics, such as superposition and entanglement, to perform complex calculations using qubits that can exist in multiple states simultaneously.


In [4]:
# @title
def make_ai_alert(coin_name, change, price, titles):

  headline_text = ""
  for title in titles:
    headline_text = headline_text+title +"\n"

  prompt = f"""
  Act as financial analyst,

  coin:{coin['name']}
  24-hour change: {change}%
  Current price: ${price}

  Recent news headlines:
  {headline_text}

  Write a professional, urgent 2-sentence Slack alert explaining the price drop using only the news conext above.
  """



  response = client.models.generate_content(
  model="gemini-2.5-flash", contents = prompt
  )
  ai_alert = response.text
  return ai_alert

In [5]:
temp = '''earth

hat'''
print(temp)

earth

hat


In [6]:
import requests
import csv

# Step 1: define the URLs
url_roster = "https://api.coingecko.com/api/v3/coins/list"
url_price = "https://api.coingecko.com/api/v3/simple/price"

#Step 2: define the target coin IDs
target_coins = ["bitcoin", "ethereum", "solana", "ripple", "dogecoin"]

#Step 3: make GET request to coin roster
roster_response = requests.get(url_roster)
roster_data = roster_response.json()
# print(roster_data)

In [7]:

# Build your alert pipelipricene below!
url_live = "https://api.coingecko.com/api/v3/simple/price"
live_response = requests.get(url_live)
live_data = live_response.json()
# print(live_data)

target_coins = ["bitcoin", "ethereum", "solana", "ripple", "dogecoin"]

price_params= {'ids': 'bitcoin,ethereum,solana,ripple,dogecoin',
               'vs_currencies': 'usd',
                'include_24hr_change':True
               }



price_response = requests.get(url_live, params=price_params)

price_data = price_response.json()

print(price_data)

{'bitcoin': {'usd': 78663, 'usd_24h_change': 0.44505360525862725}, 'dogecoin': {'usd': 0.108434, 'usd_24h_change': 0.07373304761843774}, 'ethereum': {'usd': 2329.37, 'usd_24h_change': 1.0318094345826907}, 'ripple': {'usd': 1.39, 'usd_24h_change': 0.3144703329968102}, 'solana': {'usd': 84.13, 'usd_24h_change': 0.20917181285474967}}


In [8]:
filter_coins=[]
for el in roster_data:
  # print(el)
  if el['id'] in target_coins:
    filter_coins.append(el)
print(filter_coins)


[{'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin'}, {'id': 'dogecoin', 'symbol': 'doge', 'name': 'Dogecoin'}, {'id': 'ethereum', 'symbol': 'eth', 'name': 'Ethereum'}, {'id': 'ripple', 'symbol': 'xrp', 'name': 'XRP'}, {'id': 'solana', 'symbol': 'sol', 'name': 'Solana'}]


In [13]:
from os import times_result
#inner loop to cross reference the filter roster with the live pricing data
#match through their id information
filter_price=[]
alerts= []
threshold = 0.2
for coin in filter_coins:
  # print(coin)
  for price_id in price_data:
    # print(price_id)
    if coin['id'] == price_id:
      change = price_data[price_id]['usd_24h_change']
      price = price_data[price_id]['usd']
      filter_price.append([coin['name'], coin['symbol'], price_data[price_id]['usd']]) #using list

      if change <= threshold:
        time.sleep(2)
        coin_symbol = coin["symbol"].upper()
        news_url = f"https://min-api.cryptocompare.com/data/v2/news/?lang=EN&categories={coin_symbol}"
        news_response = requests.get(news_url, headers = headers)
        news_data = news_response.json()
        top_3_news = []
        number_of_news_items = min(3,len(news_data["Data"]))
        for i in range(number_of_news_items):
          top_3_news.append(news_data["Data"][i]["title"])
        ai_alert = make_ai_alert(coin["name"],change, price, top_3_news)
        print(ai_alert)
        alerts.append(ai_alert)



Urgent DOGE Update: The price is experiencing significant downward pressure as a Dogecoin whale's $3.87 million loss critically tests the $0.10 support level. This scenario raises immediate concerns about the survival of key support, potentially leading to further depreciation.


In [10]:
for alert in alerts:
  print(alert)
  print("#########")

In [11]:
lis= ["hello","world","!!!"]
to = ""
for l in lis:
  to = to + " " +l
  print(to)


 hello
 hello world
 hello world !!!


In [14]:
with open('market_alerts.txt', "w") as file:
  if len(alerts) == 0:
    file.write('Market is stable today.')
  else:
    for alert in alerts:
      file.write(alert)
      if alert not in alerts[-1]:
        file.write("\n")


print("market_alerts.txt is imported successfully.")

market_alerts.txt is imported successfully.
